Source :

https://www.geeksforgeeks.org/implement-your-own-word2vecskip-gram-model-in-python/

In [ ]:
import numpy as np
import string
from nltk.corpus import stopwords

In [ ]:
def softmax(x):
	"""Compute softmax values for each sets of scores in x."""
	e_x = np.exp(x - np.max(x))
	return e_x / e_x.sum()

Input

One-hot

↓

W

(V,N)

↓

Embedding

N

↓

W1

(N,V)

↓

Scores

V

↓

Softmax

↓

Probabilities

model

├── N = 10
├── V = 5000
├── W = (5000 × 10)
├── W1 = (10 × 5000)
├── words = [...]
└── word_index = {...}

          initialize()

             │
             ▼
      Build Vocabulary
             │
             ▼
      Create W (Embeddings)
             │
             ▼
      Create W1 (Output Layer)
             │
             ▼
      Build Dictionary
             │
             ▼
      Ready for Training

Input Word

↓

One-hot Vector (V×1)

↓

Wᵀ

↓

Embedding (N×1)

↓

W1ᵀ

↓

Scores (V×1)

↓

Softmax

↓

Probability of Every Word

| Matrix | Shape        | Purpose                                         | Stores                                      |
| ------ | ------------ | ----------------------------------------------- | ------------------------------------------- |
| **W**  | (V \times N) | Maps an input word to its embedding             | Input word vectors (embeddings)             |
| **W1** | (N \times V) | Maps an embedding to scores over the vocabulary | Output weights for predicting context words |


| Code                    | Mathematical Meaning              | Purpose                                              |
| ----------------------- | --------------------------------- | ---------------------------------------------------- |
| `e = y - t`             | (\frac{\partial L}{\partial u})   | Error at the output (gradient of loss w.r.t. logits) |
| `dLdW1 = h @ e.T`       | (\frac{\partial L}{\partial W_1}) | Gradient for the output weight matrix                |
| `W1 @ e`                | (\frac{\partial L}{\partial h})   | Propagate the error back to the hidden layer         |
| `dLdW = X @ (W1 @ e).T` | (\frac{\partial L}{\partial W})   | Gradient for the input embedding matrix              |
| `W1 -= α·dLdW1`         | Gradient descent                  | Update output weights                                |
| `W -= α·dLdW`           | Gradient descent                  | Update input embeddings                              |


In [ ]:
class word2vec(object):
	def __init__(self):
		self.N = 500#Embedding dimension
		self.X_train = []
		self.y_train = []
		self.window_size = 2#immediate neighbors
		self.alpha = 0.001
		self.words = []
		self.word_index = {}

	def initialize(self,V,data):
		self.V = V
		self.W = np.random.uniform(-0.6, 0.4, (self.V, self.N))
		self.W1 = np.random.uniform(-0.6, 0.4, (self.N, self.V))

		self.words = data
		for i in range(len(data)):
			self.word_index[data[i]] = i


	def feed_forward(self,X):
		self.h = np.dot(self.W.T,X).reshape(self.N,1)
		self.u = np.dot(self.W1.T,self.h)
		#print(self.u)
		self.y = softmax(self.u)
		return self.y

	def backpropagate(self,x,t):
		e = self.y - np.asarray(t).reshape(self.V,1)#Loss
		# e.shape is V x 1
		dLdW1 = np.dot(self.h,e.T)
		X = np.array(x).reshape(self.V,1)
		dLdW = np.dot(X, np.dot(self.W1,e).T)
		self.W1 -=  self.alpha*dLdW1
		self.W -= self.alpha*dLdW

	def train(self,epochs):
		for x in range(1,epochs+1):
			self.loss = 0
			for j in range(len(self.X_train)):
				self.feed_forward(self.X_train[j])
				self.backpropagate(self.X_train[j],self.y_train[j])
				C = 0
				for m in range(self.V):
					if(self.y_train[j][m]):
						self.loss += -1*self.u[m]
						C += 1
				self.loss += C*np.log(np.sum(np.exp(self.u)))
			print("epoch ",x, " loss = ",self.loss)
			self.alpha *= 1/( (1+self.alpha*x) )

	def predict(self,word,number_of_predictions):
		if word in self.words:
			index = self.word_index[word]
			X = [0 for i in range(self.V)]
			X[index] = 1
			prediction = self.feed_forward(X).flatten()

			top_indices = np.argsort(prediction)[::-1][:number_of_predictions]

			return [self.words[i] for i in top_indices]
		else:
			print("Word not found in dicitonary")

In [ ]:
sen="<=>?&&&&&&&&& c  @[\]^_  &&hello!! how are yoit634 $$ ))))))"
print(sen.strip(string.punctuation))#only remove from start and the end of the sentence

 c  @[\]^_  &&hello!! how are yoit634 $$ 


<>:1: SyntaxWarning: invalid escape sequence '\]'
<>:1: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipykernel_28648/3034761183.py:1: SyntaxWarning: invalid escape sequence '\]'
  sen="<=>?&&&&&&&&& c  @[\]^_  &&hello!! how are yoit634 $$ ))))))"


In [ ]:
def preprocessing(corpus):
	stop_words = set(stopwords.words('english'))
	training_data = []
	sentences = corpus.split(".")
	for i in range(len(sentences)):
		sentences[i] = sentences[i].strip()
		sentence = sentences[i].split()
		x = [word.strip(string.punctuation).lower() for word in sentence if word not in stop_words]

		training_data.append(x)
	return training_data

In [ ]:
def prepare_data_for_training(sentences,w2v):
	data = {}
	for sentence in sentences:
		for word in sentence:
			if word not in data:
				data[word] = 1
			else:
				data[word] += 1
	V = len(data)
	data = sorted(list(data.keys()))
	vocab = {}
	for i in range(len(data)):
		vocab[data[i]] = i

	for sentence in sentences:
		for i in range(len(sentence)):
			center_word = [0 for x in range(V)]
			center_word[vocab[sentence[i]]] = 1
			context = [0 for x in range(V)]

			for j in range(i-w2v.window_size,i+w2v.window_size):
				if i!=j and j>=0 and j<len(sentence):
					context[vocab[sentence[j]]] += 1
			w2v.X_train.append(center_word)
			w2v.y_train.append(context)
	w2v.initialize(V,data)

	return w2v.X_train,w2v.y_train

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
corpus = "The earth revolves around the sun. The moon revolves around the earth"
epochs = 1000

In [ ]:
training_data = preprocessing(corpus)

In [ ]:
training_data

[['the', 'earth', 'revolves', 'around', 'sun'],
 ['the', 'moon', 'revolves', 'around', 'earth']]

In [ ]:
w2v = word2vec()

In [ ]:
prepare_data_for_training(training_data,w2v)

([[0, 0, 0, 0, 0, 1],
  [0, 1, 0, 0, 0, 0],
  [0, 0, 0, 1, 0, 0],
  [1, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 1, 0],
  [0, 0, 0, 0, 0, 1],
  [0, 0, 1, 0, 0, 0],
  [0, 0, 0, 1, 0, 0],
  [1, 0, 0, 0, 0, 0],
  [0, 1, 0, 0, 0, 0]],
 [[0, 1, 0, 0, 0, 0],
  [0, 0, 0, 1, 0, 1],
  [1, 1, 0, 0, 0, 1],
  [0, 1, 0, 1, 1, 0],
  [1, 0, 0, 1, 0, 0],
  [0, 0, 1, 0, 0, 0],
  [0, 0, 0, 1, 0, 1],
  [1, 0, 1, 0, 0, 1],
  [0, 1, 1, 1, 0, 0],
  [1, 0, 0, 1, 0, 0]])

In [ ]:
w2v.train(epochs)

epoch  1  loss =  [39.95975584]
epoch  2  loss =  [39.80691361]
epoch  3  loss =  [39.65576301]
epoch  4  loss =  [39.50643548]
epoch  5  loss =  [39.35905485]
epoch  6  loss =  [39.21373677]
epoch  7  loss =  [39.07058823]
epoch  8  loss =  [38.92970713]
epoch  9  loss =  [38.79118201]
epoch  10  loss =  [38.65509185]
epoch  11  loss =  [38.521506]
epoch  12  loss =  [38.39048415]
epoch  13  loss =  [38.26207645]
epoch  14  loss =  [38.13632369]
epoch  15  loss =  [38.01325754]
epoch  16  loss =  [37.8929009]
epoch  17  loss =  [37.77526824]
epoch  18  loss =  [37.66036607]
epoch  19  loss =  [37.54819337]
epoch  20  loss =  [37.43874211]
epoch  21  loss =  [37.33199773]
epoch  22  loss =  [37.22793971]
epoch  23  loss =  [37.12654206]
epoch  24  loss =  [37.02777386]
epoch  25  loss =  [36.93159981]
epoch  26  loss =  [36.83798067]
epoch  27  loss =  [36.74687378]
epoch  28  loss =  [36.65823357]
epoch  29  loss =  [36.5720119]
epoch  30  loss =  [36.48815856]
epoch  31  loss =  [36.

In [ ]:
w2v.predict("around",3)

['earth', 'revolves', 'moon']

In [ ]:
w2v.predict("around",5)

['earth', 'revolves', 'moon', 'around', 'the']

In [ ]:
w2v.predict("sun",3)

['around', 'revolves', 'moon']

In [ ]:
w2v.predict("earth",3)

['revolves', 'the', 'around']

In [ ]:
w2v.predict("jupiter",3)

Word not found in dicitonary
